# WIPO GII indicator eye-test — before CHIPS inclusion

Fast, reproducible plausibility and influence screen of the GII columns pulled into
`data/WIPO Global Innovation Index/gii_indicators_side2026_countries.csv`.

Scope (agreed): **all 20 numeric columns** = 18 indicators (13 fetched + 5 derived)
+ 2 denominators. Each indicator is evaluated on its **latest available year**
(the long panel is not used). For the CHIPS-impact test each indicator is attached
as a **shadow sub-pillar** at equal sub-pillar weight, under **Absolute and Relative**
and **min-max and z-score**. Raw and normalised versions are both assessed.

Outputs are written to `deliverables/gii-eye-test/`.

## Setup and data assembly

In [1]:
import sys, copy, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, "SIDE_dashboard")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown

from core.loader import load_app_data
from core import chips, chips_hierarchy as H, scaling
from components import country_names

pio.renderers.default = "notebook"
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 60)

OUT = Path("deliverables/gii-eye-test")
OUT.mkdir(parents=True, exist_ok=True)

GII_PATH = "data/WIPO Global Innovation Index/gii_indicators_side2026_countries.csv"
gii_raw = pd.read_csv(GII_PATH, dtype=str, keep_default_na=False)
DATASETS = {
    "Absolute": load_app_data("data/SIDE 2026 - Absolute.csv"),
    "Relative": load_app_data("data/SIDE 2026 - Relative.csv"),
}

iso_to_country = {country_names.COUNTRY_TO_ISO3[c]: c for c in DATASETS["Absolute"].country_list}
gii_raw["dashboard_country"] = gii_raw["iso3"].map(iso_to_country)
assert gii_raw["dashboard_country"].notna().all(), gii_raw.loc[gii_raw["dashboard_country"].isna(), "iso3"].tolist()
assert gii_raw["iso3"].is_unique and len(gii_raw) == 71

def num(col):
    return pd.to_numeric(gii_raw[col], errors="coerce")

print(f"GII wide file: {gii_raw.shape[0]} countries x {gii_raw.shape[1]} columns")
print("All 71 GII ISO-3 codes map to a dashboard country (join is on iso3, not name).")

2026-10-17 21:01:50.217 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-17 21:01:51.396 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-17 21:01:51.397 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-10-17 21:01:51.524 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


GII wide file: 71 countries x 41 columns
All 71 GII ISO-3 codes map to a dashboard country (join is on iso3, not name).


## Column inventory and placements

The 18 indicators are grouped into fetched vs derived. Each gets the **most
plausible** CHIPS home; the shadow sub-pillar sits in that pillar at equal
sub-pillar weight. The 2 denominators are assessed for coverage/quality but are
not candidates themselves.

In [2]:
# col, label, kind, year source, pillar, sub-pillar, units, domain bounds
CANDIDATES = [
    ("market_cap_pct_gdp", "Market capitalisation, % GDP", "fetched", "market_cap_pct_gdp_year",
     "INNOVATE", "Investments and Startups", "% GDP", (0, 300)),
    ("mht_manufacturing_pct", "Medium/high-tech manufacturing, % MVA", "fetched", "mht_manufacturing_pct_year",
     "HARNESS", "Real Economy", "% manufacturing VA", (0, 100)),
    ("fdi_net_inflows_pct_gdp", "FDI net inflows, % GDP", "fetched", "fdi_net_inflows_pct_gdp_year",
     "INNOVATE", "Investments and Startups", "% GDP", (-40, 50)),
    ("gerd_pct_gdp", "Gross R&D expenditure, % GDP", "fetched", "gerd_pct_gdp_year",
     "INNOVATE", "Investments and Startups", "% GDP", (0, 10)),
    ("patent_apps_residents", "Resident patent applications (count)", "fetched", "patent_apps_residents_year",
     "INNOVATE", "Investments and Startups", "count", (0, None)),
    ("sci_tech_articles_wdi", "Scientific & technical articles (count)", "fetched", "sci_tech_articles_wdi_year",
     "INNOVATE", "AI", "count", (0, None)),
    ("low_carbon_share_primary_energy_pct", "Low-carbon share of primary energy, %", "fetched", "low_carbon_share_primary_energy_pct_year",
     "SUSTAINABILITY", "Sustainability", "%", (0, 100)),
    ("gdp_per_unit_energy_use", "GDP per unit of energy use", "fetched", "gdp_per_unit_energy_use_year",
     "SUSTAINABILITY", "Sustainability", "PPP$/kgoe", (0, None)),
    ("github_git_pushes_2025", "GitHub git pushes 2025 (count)", "fetched", "fixed:2025",
     "HARNESS", "Apps and platform", "count", (0, None)),
    ("openalex_articles_2024", "OpenAlex articles 2024 (count)", "fetched", "fixed:2024",
     "INNOVATE", "AI", "count", (0, None)),
    ("openalex_company_articles_2024", "OpenAlex company-authored articles 2024", "fetched", "fixed:2024",
     "INNOVATE", "AI", "count", (0, None)),
    ("ict_service_exports_usd", "ICT service exports, USD", "fetched", "ict_service_exports_usd_year",
     "HARNESS", "Real Economy", "USD", (0, None)),
    ("ip_receipts_usd", "IP receipts, USD", "fetched", "ip_receipts_usd_year",
     "INNOVATE", "Investments and Startups", "USD", (0, None)),
    ("patents_per_bn_ppp_gdp", "Resident patents per bn PPP$ GDP", "derived", "parent:patent_apps_residents",
     "INNOVATE", "Investments and Startups", "per bn PPP$ GDP", (0, None)),
    ("sci_articles_wdi_per_bn_ppp_gdp", "Articles per bn PPP$ GDP", "derived", "parent:sci_tech_articles_wdi",
     "INNOVATE", "AI", "per bn PPP$ GDP", (0, None)),
    ("github_git_pushes_per_mn_pop_2025", "GitHub pushes per mn population", "derived", "fixed:2025",
     "HARNESS", "Apps and platform", "per mn pop", (0, None)),
    ("openalex_articles_per_bn_ppp_gdp_2024", "OpenAlex articles per bn PPP$ GDP", "derived", "fixed:2024",
     "INNOVATE", "AI", "per bn PPP$ GDP", (0, None)),
    ("openalex_company_share_pct_2024", "Company-authored article share, %", "derived", "fixed:2024",
     "INNOVATE", "AI", "%", (0, 100)),
]
DENOMINATORS = [
    ("population", "Total population", "population_year"),
    ("gdp_ppp_current_usd", "GDP, PPP, current USD", "gdp_ppp_current_usd_year"),
]

CAND_DICT = {c[0]: c for c in CANDIDATES}
DENOM_DICT = {c[0]: c for c in DENOMINATORS}
print(f"{len(CANDIDATES)} candidate indicators + {len(DENOMINATORS)} denominators = {len(CANDIDATES)+len(DENOMINATORS)} numeric columns")

inv = pd.DataFrame([
    {"Column": c[0], "Indicator": c[1], "Kind": c[2], "Placement": f"{c[5]} · {c[6]}",
     "Units": c[6], "Bounds": c[7]}
    for c in CANDIDATES
])
display(inv)

18 candidate indicators + 2 denominators = 20 numeric columns


,Column,Indicator,Kind,Placement,Units,Bounds
0,market_cap_pct_gdp,"Market capitalisation, % GDP",fetched,Investments and Startups · % GDP,% GDP,"(0, 300)"
1,mht_manufacturing_pct,"Medium/high-tech manufacturing, % MVA",fetched,Real Economy · % manufacturing VA,% manufacturing VA,"(0, 100)"
2,fdi_net_inflows_pct_gdp,"FDI net inflows, % GDP",fetched,Investments and Startups · % GDP,% GDP,"(-40, 50)"
3,gerd_pct_gdp,"Gross R&D expenditure, % GDP",fetched,Investments and Startups · % GDP,% GDP,"(0, 10)"
4,patent_apps_residents,Resident patent applications (count),fetched,Investments and Startups · count,count,"(0, None)"
5,sci_tech_articles_wdi,Scientific & technical articles (count),fetched,AI · count,count,"(0, None)"
6,low_carbon_share_primary_energy_pct,"Low-carbon share of primary energy, %",fetched,Sustainability · %,%,"(0, 100)"
7,gdp_per_unit_energy_use,GDP per unit of energy use,fetched,Sustainability · PPP$/kgoe,PPP$/kgoe,"(0, None)"
8,github_git_pushes_2025,GitHub git pushes 2025 (count),fetched,Apps and platform · count,count,"(0, None)"
9,openalex_articles_2024,OpenAlex articles 2024 (count),fetched,AI · count,count,"(0, None)"


## Section 1 — Coverage, vintage and missingness

In [3]:
def year_for(cand):
    src = cand[3]
    col = cand[0]
    if src.startswith("fixed:"):
        return pd.Series(int(src.split(":")[1]), index=gii_raw.index)
    if src.startswith("parent:"):
        parent = src.split(":")[1]
        ycol = CAND_DICT.get(parent, (None,)*4)[3]
        if ycol is None or not ycol.endswith("_year"):
            return pd.Series(np.nan, index=gii_raw.index)
        return num(ycol)
    return num(src)

REF_YEAR = 2025
cov_rows = []
for c in CANDIDATES:
    s = num(c[0]); yr = year_for(c)
    missing = gii_raw.loc[s.isna(), "country"].tolist()
    stale = gii_raw.loc[yr < REF_YEAR - 2, "country"].tolist()
    cov_rows.append({
        "Column": c[0], "Indicator": c[1], "Kind": c[2],
        "n": int(s.notna().sum()), "Coverage": round(s.notna().mean(), 3),
        "Year (mode)": int(yr.mode().iloc[0]) if yr.notna().any() else np.nan,
        "Year min": int(yr.min()) if yr.notna().any() else np.nan,
        "Stale (>2y)": len(stale),
        "Missing countries": "; ".join(missing) if missing else "",
        "Stale countries": "; ".join(stale) if stale else "",
    })
coverage = pd.DataFrame(cov_rows)
display(coverage)

# Denominators
den_rows = []
for col, label, ycol in DENOMINATORS:
    s = num(col); yr = num(ycol)
    den_rows.append({"Column": col, "Label": label, "n": int(s.notna().sum()),
                     "Coverage": round(s.notna().mean(), 3),
                     "Year min": int(yr.min()), "Year max": int(yr.max()),
                     "Zero/negative": int((s <= 0).sum()),
                     "Missing": "; ".join(gii_raw.loc[s.isna(), "country"].tolist())})
display(pd.DataFrame(den_rows))

,Column,Indicator,Kind,n,Coverage,Year (mode),Year min,Stale (>2y),Missing countries,Stale countries
0,market_cap_pct_gdp,"Market capitalisation, % GDP",fetched,62,0.873,2025,2014,13,Sweden; Denmark; Finland; Serbia; Guatemala; Dominican Republic; E...,United Kingdom; France; Ireland; Netherlands; Norway; Portugal; Be...
1,mht_manufacturing_pct,"Medium/high-tech manufacturing, % MVA",fetched,70,0.986,2024,2024,0,Dominican Republic,
2,fdi_net_inflows_pct_gdp,"FDI net inflows, % GDP",fetched,71,1.000,2025,2024,0,,
3,gerd_pct_gdp,"Gross R&D expenditure, % GDP",fetched,67,0.944,2023,2014,16,Morocco; Ghana; Bangladesh; Dominican Republic,Singapore; India; United Arab Emirates; Australia; Chile; Qatar; M...
4,patent_apps_residents,Resident patent applications (count),fetched,71,1.000,2024,2020,1,,Nigeria
5,sci_tech_articles_wdi,Scientific & technical articles (count),fetched,71,1.000,2023,2023,0,,
6,low_carbon_share_primary_energy_pct,"Low-carbon share of primary energy, %",fetched,62,0.873,2025,2025,0,Costa Rica; Serbia; Nigeria; Kenya; Ghana; Guatemala; Dominican Re...,
7,gdp_per_unit_energy_use,GDP per unit of energy use,fetched,71,1.000,2024,2024,0,,
8,github_git_pushes_2025,GitHub git pushes 2025 (count),fetched,71,1.000,2025,2025,0,,
9,openalex_articles_2024,OpenAlex articles 2024 (count),fetched,71,1.000,2024,2024,0,,


,Column,Label,n,Coverage,Year min,Year max,Zero/negative,Missing
0,population,Total population,71,1.0,2025,2025,0,
1,gdp_ppp_current_usd,"GDP, PPP, current USD",71,1.0,2024,2025,0,


## Section 2 — Distribution, plausibility and outlier scan

Rules: domain bounds; MAD robust z (|z|>3.5); Tukey fences; `max/median`;
skew; and a size-artifact check against log GDP and log population (Spearman).
A high `max/median` or skew means min-max will be dominated by the top country.

In [4]:
def sp(a, b):
    idx = a.dropna().index.intersection(b.dropna().index)
    if len(idx) < 3:
        return np.nan
    return float(a.loc[idx].rank().corr(b.loc[idx].rank()))

def mad_z(s):
    s = s.dropna()
    if len(s) < 5:
        return pd.Series(dtype=float)
    med = s.median()
    mad = (s - med).abs().median()
    if mad == 0:
        return pd.Series(0.0, index=s.index)
    return 0.6745 * (s - med) / mad

def tukey_flags(s):
    s = s.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)

gdp = num("gdp_ppp_current_usd")
pop = num("population")
log_gdp, log_pop = np.log(gdp.replace(0, np.nan)), np.log(pop.replace(0, np.nan))

def top_bottom(s, k=3):
    t = s.nlargest(k); b = s.nsmallest(k)
    return ("; ".join(f"{gii_raw.loc[i,'country']} {s.loc[i]:.4g}" for i in t.index),
            "; ".join(f"{gii_raw.loc[i,'country']} {s.loc[i]:.4g}" for i in b.index))

dist_rows = []
outlier_rows = []
for c in CANDIDATES:
    col = c[0]; s = num(col); lo, hi = c[7]
    madz = mad_z(s); tk = tukey_flags(s)
    viol = []
    if lo is not None and (s < lo).any():
        viol.append(f"<{lo}: " + "; ".join(f"{gii_raw.loc[i,'country']}={s.loc[i]:.4g}" for i in s[s < lo].index))
    if hi is not None and (s > hi).any():
        viol.append(f">{hi}: " + "; ".join(f"{gii_raw.loc[i,'country']}={s.loc[i]:.4g}" for i in s[s > hi].index))
    mx, mn = float(s.max()), float(s.median())
    size_r_gdp = sp(s, log_gdp)
    size_r_pop = sp(s, log_pop)
    top3, bot3 = top_bottom(s)
    dist_rows.append({
        "Column": col, "n": int(s.notna().sum()),
        "Min": mx if False else round(float(s.min()), 4), "Median": round(mn, 4),
        "Max": round(mx, 4), "P1": round(float(s.quantile(.01)), 4),
        "P99": round(float(s.quantile(.99)), 4),
        "Skew": round(float(s.skew()), 2), "Kurt": round(float(s.kurt()), 1),
        "Max/Median": round(mx / mn, 2) if mn and mn > 0 else np.nan,
        "MAD>3.5": int((madz.abs() > 3.5).sum()),
        "Tukey out": int(tk.sum()),
        "Bounds issues": " | ".join(viol) if viol else "",
        "Size r(GDP)": round(size_r_gdp, 2) if pd.notna(size_r_gdp) else np.nan,
        "Size r(Pop)": round(size_r_pop, 2) if pd.notna(size_r_pop) else np.nan,
        "Top 3": top3, "Bottom 3": bot3,
    })
    for i in s[s.notna()].index:
        if (i in madz.index and abs(madz.loc[i]) > 3.5) or (i in tk.index and tk.loc[i]):
            outlier_rows.append({
                "Column": col, "Country": gii_raw.loc[i, "country"], "Value": s.loc[i],
                "MAD z": round(float(madz.loc[i]), 2) if i in madz.index else np.nan,
                "Tukey": bool(tk.loc[i]) if i in tk.index else False,
            })
dist = pd.DataFrame(dist_rows)
outliers = pd.DataFrame(outlier_rows)
display(dist)
print("Outlier flags written to gii_outliers.csv:", len(outliers))

,Column,n,Min,Median,Max,P1,P99,Skew,Kurt,Max/Median,MAD>3.5,Tukey out,Bounds issues,Size r(GDP),Size r(Pop),Top 3,Bottom 3
0,market_cap_pct_gdp,62,1.909000e-01,4.251470e+01,3.272427e+02,1.992100e+00,2.904466e+02,1.62,2.4,7.70,7,4,>300: South Africa=327.2,0.41,0.09,South Africa 327.2; India 266.9; Switzerland 240.9,Algeria 0.1909; Costa Rica 3.144; Ukraine 3.373
1,mht_manufacturing_pct,70,2.693300e+00,3.379910e+01,7.755530e+01,3.419200e+00,7.310690e+01,0.14,-0.5,2.29,0,0,,0.35,-0.11,Singapore 77.56; Switzerland 71.11; Germany 61.27,Algeria 2.693; Iraq 3.745; Rwanda 7.005
2,fdi_net_inflows_pct_gdp,71,-2.837360e+01,1.667500e+00,2.634470e+01,-1.128990e+01,1.517790e+01,-1.34,23.6,15.80,4,6,,-0.18,-0.18,Singapore 26.34; Ireland 10.39; United Arab Emirates 8.262,Hungary -28.37; Belgium -3.968; Iraq -2.995
3,gerd_pct_gdp,67,6.100000e-03,1.017600e+00,6.345900e+00,2.890000e-02,5.420300e+00,1.46,2.6,6.24,2,2,,0.25,-0.19,"Israel 6.346; Korea, Rep. 4.944; Sweden 3.6",Guatemala 0.00609; Iraq 0.0407; Kuwait 0.08954
4,patent_apps_residents,71,6.000000e+00,7.430000e+02,1.672001e+06,7.400000e+00,6.908068e+05,7.76,62.9,2250.34,13,13,,0.82,0.48,China 1.672e+06; United States 2.703e+05; Japan 2.372e+05,Ghana 6; Guatemala 8; Rwanda 8
5,sci_tech_articles_wdi,71,1.045200e+02,1.355159e+04,9.327119e+05,1.075370e+02,5.814039e+05,6.10,41.5,68.83,13,11,,0.85,0.51,China 9.327e+05; United States 4.308e+05; India 2.282e+05,Dominican Republic 104.5; Guatemala 108.8; Rwanda 433.5
6,low_carbon_share_primary_energy_pct,62,1.930000e-02,1.434480e+01,6.306950e+01,6.020000e-02,6.026150e+01,1.27,1.2,4.40,3,5,,-0.12,-0.22,Sweden 63.07; Finland 58.47; France 55.38,Kuwait 0.0193; Algeria 0.0864; Singapore 0.4079
7,gdp_per_unit_energy_use,71,4.716800e+00,1.466590e+01,4.811870e+01,6.406800e+00,3.852510e+01,1.38,3.3,3.28,1,2,,0.02,-0.15,Ireland 48.12; Switzerland 34.41; Sri Lanka 31.41,Kuwait 4.717; South Africa 7.131; Uzbekistan 7.318
8,github_git_pushes_2025,71,2.057250e+05,4.094430e+06,1.398233e+08,2.192945e+05,1.244855e+08,4.69,24.1,34.15,9,9,,0.84,0.65,United States 1.398e+08; India 1.179e+08; Brazil 4.08e+07,Kuwait 2.057e+05; Israel 2.251e+05; Qatar 2.569e+05
9,openalex_articles_2024,71,6.380000e+02,2.714000e+04,9.552060e+05,8.340000e+02,7.548072e+05,4.09,18.7,35.20,12,7,,0.88,0.63,China 9.552e+05; United States 6.689e+05; Indonesia 5.542e+05,Dominican Republic 638; Guatemala 918; Rwanda 1799


Outlier flags written to gii_outliers.csv: 137


## Section 3 — Country known-fact spot check (the eye test)

Values for a fixed panel of large/known/atypical economies, plus curated notes
where the number is surprising or has a documented source caveat. This is the
human "does this make sense?" layer; the rules in Section 2 are the mechanical layer.

In [5]:
SPOT = ["United States", "China", "India", "Japan", "Germany", "Korea, Rep.",
        "Singapore", "Ireland", "Israel", "Switzerland", "Brazil", "South Africa",
        "Nigeria", "United Arab Emirates", "Guatemala", "Hungary", "Ukraine", "Indonesia"]

spot_rows = []
for _, r in gii_raw[gii_raw["country"].isin(SPOT)].iterrows():
    row = {"Country": r["country"]}
    for c in CANDIDATES:
        row[c[0]] = num(c[0]).loc[r.name]
    spot_rows.append(row)
spot = pd.DataFrame(spot_rows).set_index("Country").reindex(SPOT)

KNOWN = {
    ("market_cap_pct_gdp", "India"): "2025=266.9 vs 2024=136.4 (provenance) — implausible jump",
    ("market_cap_pct_gdp", "South Africa"): "327% GDP — unusually high; verify",
    ("gerd_pct_gdp", "Guatemala"): "0.006% GDP — implausibly low",
    ("gerd_pct_gdp", "India"): "latest year 2020 — stale",
    ("fdi_net_inflows_pct_gdp", "Hungary"): "−28% GDP — disinvestment one-off; verify",
    ("github_git_pushes_2025", "China"): "GitHub understated (firewall); per-capita 6k vs Singapore 1.6m",
    ("github_git_pushes_per_mn_pop_2025", "China"): "Understated denominator issue + firewall",
    ("openalex_articles_2024", "Indonesia"): "554k > India 306k — OpenAlex universe inflation",
    ("openalex_articles_2024", "China"): "OpenAlex universe differs from WIPO/Clarivate",
    ("ict_service_exports_usd", "Ireland"): "~358bn — multinational transfer-pricing effect; real but extreme",
    ("ip_receipts_usd", "Netherlands"): "Tax-domicile effect; real but extreme",
    ("gdp_per_unit_energy_use", "Ireland"): "GDP inflated by multinationals; efficiency overstated",
    ("sci_tech_articles_wdi", "India"): "228k (WDI 2023) — plausible; source universe differs from OpenAlex",
}

note_rows = []
for (col, ctry), note in KNOWN.items():
    val = num(col).loc[gii_raw.index[gii_raw["country"] == ctry][0]]
    note_rows.append({"Indicator": col, "Country": ctry, "Value": round(float(val), 4), "Note": note})
known_df = pd.DataFrame(note_rows)

# wide spot-check table (values only), transposed for readability
display(Markdown("**Spot-check values (latest year per indicator)**"))
display(spot.T)
display(Markdown("**Curated known-fact notes**"))
display(known_df)

**Spot-check values (latest year per indicator)**

Country,United States,China,India,Japan,Germany,"Korea, Rep.",Singapore,Ireland,Israel,Switzerland,Brazil,South Africa,Nigeria,United Arab Emirates,Guatemala,Hungary,Ukraine,Indonesia
market_cap_pct_gdp,2.240446e+02,7.954060e+01,2.669212e+02,1.715956e+02,5.738816e+01,1.472284e+02,1.365090e+02,2.783220e+01,6.101630e+01,2.409426e+02,3.822224e+01,3.272427e+02,2.989836e+01,1.908591e+02,NaN,2.539186e+01,3.373369e+00,6.568610e+01
mht_manufacturing_pct,5.296629e+01,4.145127e+01,4.540849e+01,5.819712e+01,6.126990e+01,6.120614e+01,7.755533e+01,5.451147e+01,4.853152e+01,7.110833e+01,3.193202e+01,1.293380e+01,3.343626e+01,4.371805e+01,2.240264e+01,5.433349e+01,3.100302e+01,3.215007e+01
fdi_net_inflows_pct_gdp,9.977930e-01,4.101940e-01,1.031117e+00,1.032786e+00,1.884623e+00,8.440670e-01,2.634473e+01,1.039207e+01,4.296505e+00,1.523000e-03,3.406969e+00,-5.058230e-01,1.377386e+00,8.261773e+00,1.390276e+00,-2.837364e+01,7.575850e-01,1.570109e+00
gerd_pct_gdp,3.447160e+00,2.577290e+00,6.455800e-01,3.441250e+00,3.153900e+00,4.943520e+00,1.808210e+00,1.588290e+00,6.345890e+00,3.221780e+00,1.193800e+00,6.147900e-01,2.844400e-01,1.494690e+00,6.090000e-03,1.377480e+00,3.698800e-01,2.806800e-01
patent_apps_residents,2.702950e+05,1.672001e+06,6.321700e+04,2.371690e+05,4.008500e+04,1.957860e+05,1.898000e+03,6.800000e+01,1.448000e+03,1.191000e+03,5.752000e+03,3.500000e+02,4.100000e+02,2.780000e+02,8.000000e+00,4.800000e+02,9.690000e+02,1.931000e+03
sci_tech_articles_wdi,4.308433e+05,9.327119e+05,2.281743e+05,9.654344e+04,1.087823e+05,7.172433e+04,1.189360e+04,8.719610e+03,1.384424e+04,2.280386e+04,5.829204e+04,1.494903e+04,9.799390e+03,7.075140e+03,1.088300e+02,8.260490e+03,1.549836e+04,3.899200e+04
low_carbon_share_primary_energy_pct,1.682920e+01,1.245440e+01,7.006900e+00,1.432150e+01,1.516530e+01,1.873020e+01,4.079000e-01,1.401240e+01,5.435100e+00,4.338080e+01,3.768440e+01,4.731800e+00,NaN,9.525100e+00,NaN,2.788820e+01,2.778390e+01,1.168050e+01
gdp_per_unit_energy_use,1.172954e+01,8.090613e+00,1.212569e+01,1.580435e+01,2.179758e+01,9.897261e+00,2.564978e+01,4.811867e+01,1.939826e+01,3.441358e+01,1.285965e+01,7.131084e+00,2.519255e+01,7.886753e+00,1.345001e+01,1.616309e+01,9.320253e+00,1.418915e+01
github_git_pushes_2025,1.398233e+08,8.536391e+06,1.179122e+08,2.358562e+07,3.128236e+07,2.779759e+07,9.776570e+06,2.693338e+06,2.251100e+05,6.407702e+06,4.079950e+07,3.768019e+06,7.862088e+06,2.168292e+06,7.911670e+05,2.224895e+06,7.675422e+06,1.996745e+07
openalex_articles_2024,6.689220e+05,9.552060e+05,3.056600e+05,1.436090e+05,1.697570e+05,9.877700e+04,2.428100e+04,1.849100e+04,1.954000e+03,5.059900e+04,2.131860e+05,3.464600e+04,4.780500e+04,1.648400e+04,9.180000e+02,1.911300e+04,8.024500e+04,5.542000e+05


**Curated known-fact notes**

,Indicator,Country,Value,Note
0,market_cap_pct_gdp,India,2.669212e+02,2025=266.9 vs 2024=136.4 (provenance) — implausible jump
1,market_cap_pct_gdp,South Africa,3.272427e+02,327% GDP — unusually high; verify
2,gerd_pct_gdp,Guatemala,6.100000e-03,0.006% GDP — implausibly low
3,gerd_pct_gdp,India,6.456000e-01,latest year 2020 — stale
4,fdi_net_inflows_pct_gdp,Hungary,-2.837360e+01,−28% GDP — disinvestment one-off; verify
5,github_git_pushes_2025,China,8.536391e+06,GitHub understated (firewall); per-capita 6k vs Singapore 1.6m
6,github_git_pushes_per_mn_pop_2025,China,6.068880e+03,Understated denominator issue + firewall
7,openalex_articles_2024,Indonesia,5.542000e+05,554k > India 306k — OpenAlex universe inflation
8,openalex_articles_2024,China,9.552060e+05,OpenAlex universe differs from WIPO/Clarivate
9,ict_service_exports_usd,Ireland,3.587607e+11,~358bn — multinational transfer-pricing effect; real but extreme


## Section 4 — Construct validity and size artifacts

- **Construct r**: Spearman of the candidate with the *existing* target-pillar
  score. Low/negative values suggest it measures something else.
- **Size r**: already in Section 2; here we highlight `|r|>0.8` as size-driven.

In [6]:
def baseline_tables(base, method):
    d = DATASETS[base]
    pillars, _ = H.resolve_hierarchy(d.numeric_df.columns)
    return d, pillars, chips.chips_table(d, pillars=pillars, method=method)

construct_rows = []
for base in ["Absolute", "Relative"]:
    d, pillars, tab = baseline_tables(base, scaling.METHOD_Z)
    for c in CANDIDATES:
        s = num(c[0])
        series = pd.Series(d.numeric_df["Country"].map(dict(zip(gii_raw["dashboard_country"], s))).values,
                           index=d.numeric_df.index)
        pillar_score = tab.set_index("Country")[c[4]]
        pair = pd.DataFrame({"x": series, "y": series.index.map(lambda i: pillar_score.get(d.numeric_df.loc[i, "Country"]))})
        r = sp(pair["x"], pair["y"])
        construct_rows.append({"Base": base, "Column": c[0], "Target pillar": c[4],
                               "Construct r": round(float(r), 3) if pd.notna(r) else np.nan})
construct = pd.DataFrame(construct_rows)
display(construct.pivot(index="Column", columns="Base", values="Construct r").reset_index())

Base,Column,Absolute,Relative
0,fdi_net_inflows_pct_gdp,-0.092,-0.087
1,gdp_per_unit_energy_use,-0.042,0.026
2,gerd_pct_gdp,0.623,0.767
3,github_git_pushes_2025,0.749,0.279
4,github_git_pushes_per_mn_pop_2025,0.419,0.678
5,ict_service_exports_usd,0.820,0.765
6,ip_receipts_usd,0.801,0.850
7,low_carbon_share_primary_energy_pct,0.272,0.223
8,market_cap_pct_gdp,0.606,0.631
9,mht_manufacturing_pct,0.690,0.703


## Section 5 — Raw vs normalised (derived) consistency

For each parent/derived pair: how correlated are they, does the derived version
reduce the size artifact, and does it change the outlier picture?

In [7]:
PAIRS = [
    ("patent_apps_residents", "patents_per_bn_ppp_gdp"),
    ("sci_tech_articles_wdi", "sci_articles_wdi_per_bn_ppp_gdp"),
    ("github_git_pushes_2025", "github_git_pushes_per_mn_pop_2025"),
    ("openalex_articles_2024", "openalex_articles_per_bn_ppp_gdp_2024"),
    ("openalex_articles_2024", "openalex_company_share_pct_2024"),
]
pair_rows = []
for raw_col, der_col in PAIRS:
    a, b = num(raw_col), num(der_col)
    r_gdp_raw = sp(a, log_gdp)
    r_gdp_der = sp(b, log_gdp)
    pair_rows.append({
        "Raw": raw_col, "Derived": der_col,
        "Spearman(raw, derived)": round(float(sp(a, b)), 3),
        "Size r(GDP) raw": round(float(r_gdp_raw), 2),
        "Size r(GDP) derived": round(float(r_gdp_der), 2),
        "Coverage raw": round(a.notna().mean(), 2),
        "Coverage derived": round(b.notna().mean(), 2),
    })
display(pd.DataFrame(pair_rows))

,Raw,Derived,"Spearman(raw, derived)",Size r(GDP) raw,Size r(GDP) derived,Coverage raw,Coverage derived
0,patent_apps_residents,patents_per_bn_ppp_gdp,0.863,0.82,0.48,1.0,1.0
1,sci_tech_articles_wdi,sci_articles_wdi_per_bn_ppp_gdp,0.477,0.85,-0.01,1.0,1.0
2,github_git_pushes_2025,github_git_pushes_per_mn_pop_2025,0.349,0.84,0.03,1.0,1.0
3,openalex_articles_2024,openalex_articles_per_bn_ppp_gdp_2024,0.318,0.88,-0.10,1.0,1.0
4,openalex_articles_2024,openalex_company_share_pct_2024,0.182,0.88,0.08,1.0,1.0


## Section 6 — CHIPS impact as a shadow sub-pillar

For each candidate: add a new sub-pillar containing only that indicator, weighted
**equally with the existing sub-pillars in the chosen pillar** (so the indicator
gets 1/(N+1) of the pillar). Compare baseline vs with-shadow for:
Δ pillar score, Δ CHIPS score, Δ CHIPS rank — under **Absolute/Relative ×
min-max/z-score**. Also report the indicator's own concentration (top-1/top-2 gap,
median min-max score) to show whether one country dominates its 0–1 range.

In [8]:
def inject_series(d, col, values_by_country):
    nd = d.numeric_df.copy()
    nd[col] = nd["Country"].map(values_by_country)
    new = copy.copy(d)
    new.numeric_df = nd
    new.higher_is_better = dict(d.higher_is_better)
    new.higher_is_better[col] = True
    new.indicators = list(d.indicators) + [col]
    return new

def shadow_spec(pillar, col):
    spec = copy.deepcopy(H.HIERARCHY)
    for p in spec:
        if p["name"] == pillar:
            p["sub_pillars"].append({"name": f"Shadow: {col}", "indicators": [(col, 1.0)]})
            break
    return spec

def concentration(s):
    s = s.dropna()
    if len(s) < 3:
        return {}
    srt = s.sort_values(ascending=False)
    return {"top1_minus_top2": float(srt.iloc[0] - srt.iloc[1]),
            "median_score": float(s.median()),
            "share_below_025": float((s < 0.25).mean())}

infl_rows = []      # one row per candidate x base x method
country_rows = []   # long: candidate x base x method x country
BASELINE = {}
for base in DATASETS:
    for method in [scaling.METHOD_FULL, scaling.METHOD_Z]:
        BASELINE[(base, method)] = baseline_tables(base, method)

values_by_country = {c[0]: dict(zip(gii_raw["dashboard_country"], num(c[0]))) for c in CANDIDATES}

for c in CANDIDATES:
    col, label, kind, _ys, pillar, sub, units, _bounds = c
    for base in DATASETS:
        d0, pillars0, tab0 = BASELINE[(base, scaling.METHOD_FULL)]
        aug = inject_series(d0, col, values_by_country[col])
        spec = shadow_spec(pillar, col)
        pillars1, unres = H.resolve_hierarchy(aug.numeric_df.columns, spec=spec)
        assert not unres, unres
        gw = chips.leaf_global_weights(pillars1).get(col, np.nan)
        for method in [scaling.METHOD_FULL, scaling.METHOD_Z]:
            _, _, tab_base = BASELINE[(base, method)]
            tab_sh = chips.chips_table(aug, pillars=pillars1, method=method)
            j = tab_base[["Country", "chips", "rank", pillar]].merge(
                tab_sh[["Country", "chips", "rank", pillar]], on="Country", suffixes=("_b", "_s"))
            j["dpillar"] = j[f"{pillar}_s"] - j[f"{pillar}_b"]
            j["dchips"] = j["chips_s"] - j["chips_b"]
            j["drank"] = j["rank_s"] - j["rank_b"]
            top = j.reindex(j["dchips"].abs().sort_values(ascending=False).index).iloc[0]
            # indicator concentration under this base/method
            sc = chips.score_matrix(aug, pillars1, method=method)[col]
            conc = concentration(sc)
            infl_rows.append({
                "Base": base, "Method": method, "Column": col, "Pillar": pillar,
                "Shadow sub-pillar": sub, "Shadow global weight": round(float(gw), 4),
                "max |dPillar|": round(float(j["dpillar"].abs().max()), 3),
                "max |dCHIPS|": round(float(j["dchips"].abs().max()), 3),
                "max |dRank|": int(j["drank"].abs().max()),
                "#|dRank|>=3": int((j["drank"].abs() >= 3).sum()),
                "Top CHIPS mover": f"{top['Country']} ({top['dchips']:+.3f})",
                "Indicator top1-top2": round(conc.get("top1_minus_top2", np.nan), 3),
                "Indicator median score": round(conc.get("median_score", np.nan), 3),
                "Share below 0.25": round(conc.get("share_below_025", np.nan), 3),
            })
            for _, r in j.iterrows():
                country_rows.append({
                    "Base": base, "Method": method, "Column": col,
                    "Country": r["Country"], "dPillar": r["dpillar"],
                    "dCHIPS": r["dchips"], "dRank": r["drank"],
                })

infl = pd.DataFrame(infl_rows)
infl_country = pd.DataFrame(country_rows)
print("Influence rows:", len(infl), "| country-level rows:", len(infl_country))

Influence rows: 72 | country-level rows: 5112


In [9]:
# Compact pivot: worst-case influence per candidate across the four runs
worst = (infl.groupby(["Column", "Pillar"])
         .agg(max_dCHIPS=("max |dCHIPS|", "max"),
              max_dRank=("max |dRank|", "max"),
              shadow_w=("Shadow global weight", "max"),
              top_mover=("Top CHIPS mover", "first"))
         .reset_index()
         .sort_values(["max_dRank", "max_dCHIPS"], ascending=False))
display(Markdown("**Worst-case CHIPS impact across Absolute/Relative × min-max/z-score**"))
display(worst)

for base in ["Absolute", "Relative"]:
    piv = infl[infl.Base == base].pivot(index="Column", columns="Method",
                                        values="max |dRank|").reset_index()
    display(Markdown(f"**Max CHIPS rank change — {base}**"))
    display(piv.sort_values(scaling.METHOD_FULL, ascending=False))

**Worst-case CHIPS impact across Absolute/Relative × min-max/z-score**

,Column,Pillar,max_dCHIPS,max_dRank,shadow_w,top_mover
8,market_cap_pct_gdp,INNOVATE,0.065,41,0.0625,South Africa (+0.062)
7,low_carbon_share_primary_energy_pct,SUSTAINABILITY,0.041,40,0.0625,Sweden (+0.039)
11,openalex_articles_per_bn_ppp_gdp_2024,INNOVATE,0.091,38,0.0625,Ukraine (+0.061)
13,openalex_company_share_pct_2024,INNOVATE,0.086,37,0.0625,Greece (+0.061)
0,fdi_net_inflows_pct_gdp,INNOVATE,0.059,31,0.0625,Singapore (+0.059)
16,sci_articles_wdi_per_bn_ppp_gdp,INNOVATE,0.088,27,0.0625,Portugal (+0.062)
9,mht_manufacturing_pct,HARNESS,0.037,26,0.0417,Qatar (+0.037)
1,gdp_per_unit_energy_use,SUSTAINABILITY,0.041,23,0.0625,Ireland (+0.041)
5,ict_service_exports_usd,HARNESS,0.037,22,0.0417,Ireland (+0.037)
2,gerd_pct_gdp,INNOVATE,0.105,21,0.0625,Israel (+0.060)


**Max CHIPS rank change — Absolute**

Method,Column,full,z
8,market_cap_pct_gdp,41,21
7,low_carbon_share_primary_energy_pct,40,20
11,openalex_articles_per_bn_ppp_gdp_2024,38,28
13,openalex_company_share_pct_2024,37,23
0,fdi_net_inflows_pct_gdp,31,19
16,sci_articles_wdi_per_bn_ppp_gdp,27,20
9,mht_manufacturing_pct,26,20
1,gdp_per_unit_energy_use,23,19
5,ict_service_exports_usd,22,9
2,gerd_pct_gdp,21,15


**Max CHIPS rank change — Relative**

Method,Column,full,z
11,openalex_articles_per_bn_ppp_gdp_2024,24,14
16,sci_articles_wdi_per_bn_ppp_gdp,24,18
2,gerd_pct_gdp,21,15
13,openalex_company_share_pct_2024,19,13
12,openalex_company_articles_2024,19,12
10,openalex_articles_2024,19,13
17,sci_tech_articles_wdi,18,12
15,patents_per_bn_ppp_gdp,18,11
14,patent_apps_residents,18,10
6,ip_receipts_usd,18,7


## Section 7 — Scorecard, verdicts and red flags

One row per indicator: coverage, vintage, distribution flags, size artifact,
construct alignment, and worst-case CHIPS influence. The `Verdict` is a screening
label only (`OK`, `coverage`, `stale`, `outliers`, `minmax-dominated`,
`size-driven`, `bounds`); it is **not** an inclusion recommendation.

In [10]:
score = coverage[["Column", "Indicator", "Kind", "n", "Coverage", "Year (mode)", "Stale (>2y)"]].copy()
score = score.merge(dist[["Column", "Max/Median", "Skew", "MAD>3.5", "Tukey out",
                          "Bounds issues", "Size r(GDP)", "Size r(Pop)", "Top 3", "Bottom 3"]], on="Column")
score = score.merge(worst[["Column", "max_dCHIPS", "max_dRank", "top_mover"]], on="Column", how="left")
con_z = construct[construct.Base == "Absolute"].set_index("Column")["Construct r"]
score["Construct r (Abs)"] = score["Column"].map(con_z)

def verdict(r):
    flags = []
    if r["Bounds issues"]:
        flags.append("bounds")
    if r["Coverage"] < 0.85:
        flags.append("coverage")
    if r["Stale (>2y)"] > 0:
        flags.append("stale")
    if r["MAD>3.5"] >= 2:
        flags.append("outliers")
    if pd.notna(r["Max/Median"]) and r["Max/Median"] > 20:
        flags.append("minmax-dominated")
    if (pd.notna(r["Size r(GDP)"]) and abs(r["Size r(GDP)"]) > 0.8) or \
       (pd.notna(r["Size r(Pop)"]) and abs(r["Size r(Pop)"]) > 0.8):
        flags.append("size-driven")
    return "; ".join(flags) if flags else "OK"

score["Verdict"] = score.apply(verdict, axis=1)
score = score.sort_values(["Verdict", "max_dRank"], ascending=[True, False]).reset_index(drop=True)
display(score[["Column", "Placement" if "Placement" in score else "Kind", "Coverage", "Year (mode)",
               "Stale (>2y)", "Max/Median", "Skew", "MAD>3.5", "Bounds issues",
               "Size r(GDP)", "Construct r (Abs)", "max_dCHIPS", "max_dRank", "Verdict"]])

,Column,Kind,Coverage,Year (mode),Stale (>2y),Max/Median,Skew,MAD>3.5,Bounds issues,Size r(GDP),Construct r (Abs),max_dCHIPS,max_dRank,Verdict
0,sci_articles_wdi_per_bn_ppp_gdp,derived,1.000,2023,0,2.31,0.26,0,,-0.01,0.349,0.088,27,OK
1,mht_manufacturing_pct,fetched,0.986,2024,0,2.29,0.14,0,,0.35,0.690,0.037,26,OK
2,gdp_per_unit_energy_use,fetched,1.000,2024,0,3.28,1.38,1,,0.02,-0.042,0.041,23,OK
3,github_git_pushes_per_mn_pop_2025,derived,1.000,2025,0,9.25,2.65,1,,0.03,0.419,0.037,16,OK
4,market_cap_pct_gdp,fetched,0.873,2025,13,7.70,1.62,7,>300: South Africa=327.2,0.41,0.606,0.065,41,bounds; stale; outliers
5,low_carbon_share_primary_energy_pct,fetched,0.873,2025,0,4.40,1.27,3,,-0.12,0.272,0.041,40,outliers
6,openalex_articles_per_bn_ppp_gdp_2024,derived,1.000,2024,0,3.92,1.59,2,,-0.10,0.159,0.091,38,outliers
7,openalex_company_share_pct_2024,derived,1.000,2024,0,6.06,1.06,8,,0.08,0.523,0.086,37,outliers
8,fdi_net_inflows_pct_gdp,fetched,1.000,2025,0,15.80,-1.34,4,,-0.18,-0.092,0.059,31,outliers
9,ict_service_exports_usd,fetched,0.986,2025,0,74.42,5.22,12,,0.48,0.820,0.037,22,outliers; minmax-dominated


In [11]:
# Write all side outputs
coverage.to_csv(OUT / "gii_coverage.csv", index=False)
dist.to_csv(OUT / "gii_distribution.csv", index=False)
outliers.to_csv(OUT / "gii_outliers.csv", index=False)
known_df.to_csv(OUT / "gii_known_fact_notes.csv", index=False)
construct.to_csv(OUT / "gii_construct_validity.csv", index=False)
infl.to_csv(OUT / "gii_chips_influence.csv", index=False)
infl_country.to_csv(OUT / "gii_chips_influence_country.csv", index=False)
score.to_csv(OUT / "gii_scorecard.csv", index=False)
spot.T.to_csv(OUT / "gii_country_spotchecks.csv")
print("Wrote 9 files to", OUT)
print(sorted(p.name for p in OUT.glob('*.csv')))

Wrote 9 files to deliverables/gii-eye-test
['gii_chips_influence.csv', 'gii_chips_influence_country.csv', 'gii_construct_validity.csv', 'gii_country_spotchecks.csv', 'gii_coverage.csv', 'gii_distribution.csv', 'gii_known_fact_notes.csv', 'gii_outliers.csv', 'gii_scorecard.csv']


## Section 8 — How to read this

- **High correlation with log GDP/population** does not automatically disqualify an
  indicator — many digital/innovation measures genuinely scale with economy size —
  but it warns that equal-weight inclusion may mostly reward large economies.
- **`max/median` and skew** matter more for **min-max** (one endpoint sets the whole
  scale). The z-score is mean/std driven and usually less compressed, but its
  logistic map saturates the upper tail.
- **Shadow sub-pillar** weights are deliberately generous (equal to a full
  sub-pillar). If even that produces small ΔCHIPS, the indicator is low-risk for
  the composite; if it moves ranks a lot, the placement/weight needs discussion.
- **A flag is a prompt to investigate, not a rejection.** The curated notes and
  the top/bottom tables are the starting points for the deeper dig.

**No inclusion recommendation is made here.**